# 4. Đánh giá BLEU Score (SacreBLEU)

Notebook này tải các Checkpoint đã huấn luyện từ Google Drive và tiến hành dịch toàn bộ tập Test (hoặc một tập con) để tính điểm BLEU chính thức so sánh giữa Transformer và LSTM.

In [ ]:
!pip install sacrebleu tokenizers torch

In [ ]:
import torch
import torch.nn as nn
import math
import sacrebleu
from tokenizers import Tokenizer
from tqdm import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/Multilingual_MT'
except:
    BASE_DIR = 'Multilingual_MT'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Tải Tokenizer
tokenizer = Tokenizer.from_file(f"{BASE_DIR}/tokenizer/tokenizer.json")
PAD_IDX = tokenizer.token_to_id("[PAD]")
BOS_IDX = tokenizer.token_to_id("[BOS]")
EOS_IDX = tokenizer.token_to_id("[EOS]")

## 1. Tải Model Transformer

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe.transpose(0, 1)[:, :x.size(1), :]
        return self.dropout(x)

class TransformerMT(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048, dropout=0.1, pad_idx=0):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        self.transformer = nn.Transformer(d_model=d_model, nhead=nhead, num_encoder_layers=num_encoder_layers,
                                          num_decoder_layers=num_decoder_layers, dim_feedforward=dim_feedforward,
                                          dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(d_model, vocab_size)
        
    def create_mask(self, src, tgt):
        tgt_mask = (torch.triu(torch.ones((tgt.shape[1], tgt.shape[1]), device=src.device)) == 1).transpose(0, 1)
        tgt_mask = tgt_mask.float().masked_fill(tgt_mask == 0, float('-inf')).masked_fill(tgt_mask == 1, float(0.0))
        src_mask = torch.zeros((src.shape[1], src.shape[1]), device=src.device).type(torch.bool)
        src_padding_mask = (src == self.pad_idx)
        tgt_padding_mask = (tgt == self.pad_idx)
        return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = self.create_mask(src, tgt)
        src_emb = self.pos_encoder(self.embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.embedding(tgt) * math.sqrt(self.d_model))
        outs = self.transformer(src_emb, tgt_emb, src_mask=src_mask, tgt_mask=tgt_mask,
                                memory_mask=None, src_key_padding_mask=src_padding_mask, 
                                tgt_key_padding_mask=tgt_padding_mask, memory_key_padding_mask=src_padding_mask)
        return self.fc_out(outs)

print("Đang load mô hình Transformer từ Checkpoint...")
transformer_model = TransformerMT(vocab_size=tokenizer.get_vocab_size(), pad_idx=PAD_IDX).to(DEVICE)

# THAY ĐỔI SỐ EPOCH Ở ĐÂY SAU KHI TRAIN XONG
BEST_EPOCH = 10 
try:
    ckpt = torch.load(f"{BASE_DIR}/model_assets/transformer_ep{BEST_EPOCH}.pt", map_location=DEVICE)
    transformer_model.load_state_dict(ckpt['model_state'])
    transformer_model.eval()
    print("Load Transformer thành công!")
except FileNotFoundError:
    print("Chưa tìm thấy Checkpoint của Transformer. Hãy đảm bảo bạn đã train xong.")

## 2. Hàm Dịch (Greedy Decoding)

In [ ]:
def translate_transformer(model, src_text, target_tag, max_len=128):
    model.eval()
    src_full = f"{target_tag} {src_text}"
    src_ids = tokenizer.encode(src_full).ids
    src_tensor = torch.tensor([BOS_IDX] + src_ids + [EOS_IDX], dtype=torch.long).unsqueeze(0).to(DEVICE)
    
    tgt_indices = [BOS_IDX]
    
    with torch.no_grad():
        for i in range(max_len):
            tgt_tensor = torch.tensor(tgt_indices, dtype=torch.long).unsqueeze(0).to(DEVICE)
            
            output = model(src_tensor, tgt_tensor)
            # Lấy token cuối cùng của sequence
            next_token = output[0, -1, :].argmax().item()
            
            if next_token == EOS_IDX:
                break
            tgt_indices.append(next_token)
            
    # Decode ra text, bỏ đi BOS
    translated_text = tokenizer.decode(tgt_indices[1:])
    return translated_text

## 3. Tính điểm BLEU trên tập Test

In [ ]:
def evaluate_bleu(model, test_file_path, num_samples=1000):
    print(f"Tiến hành chấm điểm BLEU trên {num_samples} câu từ tập test...")
    
    with open(test_file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    import random
    random.shuffle(lines)
    
    references = []
    hypotheses = []
    
    # Giới hạn số lượng mẫu để chạy nhanh
    sample_lines = lines[:num_samples]
    
    for line in tqdm(sample_lines):
        parts = line.strip().split('\t')
        if len(parts) != 3: continue
        target_tag, src_text, ref_text = parts
        
        pred_text = translate_transformer(model, src_text, target_tag)
        
        references.append(ref_text)
        hypotheses.append(pred_text)
        
    # sacrebleu yêu cầu mảng các mảng references: [[ref1, ref2], [ref1, ref2]]
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    print("\n===========================")
    print(f"BLEU SCORE: {bleu.score:.2f}")
    print("===========================")
    
    # In thử 5 câu để xem thực tế
    print("\n--- 5 Câu mẫu ---")
    for i in range(5):
        print(f"Gốc : {sample_lines[i].split('\t')[1]}")
        print(f"Ref : {references[i]}")
        print(f"Dịch: {hypotheses[i]}")
        print("-")
        
    return bleu.score

# Chạy đánh giá (Chỉ chạy sau khi model đã load thành công)
test_file = f"{BASE_DIR}/data/processed/test.txt"
if 'transformer_model' in locals() and next(transformer_model.parameters()).device.type != 'meta':
    evaluate_bleu(transformer_model, test_file, num_samples=500)